# NicoComic YOLOX-Nano 640 panel training

Use a free Google Colab GPU runtime. This run downloads only the SHA-pinned CC0-1.0 Comix v0 tiny dataset and the public model repository. Faster R-CNN boxes are pseudo labels for training; they are not product quality truth. No private comics or annotations are uploaded.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'


In [ ]:
!git clone -q https://github.com/zyuanming/NicoComic-Reader-Models.git /content/NicoComic-Reader-Models
%cd /content/NicoComic-Reader-Models
!git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX
!git -C /content/YOLOX checkout -q 419778480ab6ec0590e5d3831b3afb3b46ab2aa3
!pip -q install -r scripts/requirements-training.txt
!pip -q install -e /content/YOLOX --no-build-isolation --no-deps


In [ ]:
!python scripts/download_comix_v0.py datasets/comix_v0_tiny_pages.json /content/comix-v0
!python scripts/prepare_comix_v0_coco.py datasets/comix_v0_tiny_pages.json /content/comix-v0-coco /content/comix-v0/*.tar


In [ ]:
!python scripts/train_yolox_panels.py experiments/yolox_nano_panels_640.py /content/comix-v0-coco /content/yolox-640 --device cuda --batch-size 8 --epochs 100 --workers 2
!zip -q -j /content/yolox-640-checkpoints.zip /content/yolox-640/epoch_*_ckpt.pth


In [ ]:
!python scripts/evaluate_yolox_coco.py experiments/yolox_nano_panels_640.py /content/yolox-640/latest_ckpt.pth /content/comix-v0-coco/annotations/instances_val2017.json /content/comix-v0-coco/val2017 /content/yolox-640-validation.json
!cat /content/yolox-640-validation.json


In [ ]:
!python scripts/export_yolox_onnx.py experiments/yolox_nano_panels_640.py /content/yolox-640/latest_ckpt.pth /content/NicoComicPanelYOLOXNano640.onnx
from google.colab import files
files.download('/content/NicoComicPanelYOLOXNano640.onnx')
files.download('/content/yolox-640-validation.json')
files.download('/content/yolox-640-checkpoints.zip')


After downloading the ONNX file, convert it to Core ML on macOS and run the unchanged private 60-panel/20-fallback scorer. Do not publish or integrate the model based on the pseudo-label validation score.
